# Lab Day 2 — Backbone, công thức huấn luyện và suy luận trên DeepWeeds

Notebook **khung** cho Colab hoặc Kaggle. Các ô có chữ `TODO` là phần **bạn tự viết**; ô cài đặt, tải dữ liệu
và chạy `eval.py` đã viết sẵn. Đọc `README.md` (quy tắc chia dữ liệu, cách đánh giá), `GUIDE.md` và `RUBRIC.md` trước.

**Quy tắc quan trọng nhất:** chọn mọi thứ trên **val**; **test chỉ chạy một lần mỗi seed ở Bước 4**.

Bản đồ: Bước 0 (EDA, kiểm tra) → Bước 1 (backbone) → Bước 2 (công thức huấn luyện) → Bước 3 (suy luận)
→ Bước 4 (chung kết + `eval.py`) → Bước 5 (xlsx, biểu đồ, báo cáo).

## 0. Cài đặt môi trường

In [ ]:
# Colab/Kaggle normally provide CUDA-enabled torch and torchvision.
!pip -q install timm openpyxl fvcore

import os, sys, platform
from pathlib import Path
import numpy as np, pandas as pd, torch, timm

# Colab-friendly root detection: mount Drive before running from a repo under MyDrive.
try:
    from google.colab import drive
    drive.mount('/content/drive')
    drive_root = Path('/content/drive')
    ROOT = None
    for base in [drive_root / 'MyDrive', drive_root]:
        probe = base
        while probe != probe.parent and not (probe / 'eval.py').exists():
            probe = probe.parent
        if (probe / 'eval.py').exists():
            ROOT = probe
            break
    if ROOT is None:
        raise FileNotFoundError('Clone the repo to Google Drive before running in Colab.')
except Exception:
    ROOT = Path.cwd()
    while not (ROOT / 'eval.py').exists() and ROOT != ROOT.parent:
        ROOT = ROOT.parent
if not (ROOT / 'eval.py').exists():
    raise FileNotFoundError("Open this notebook from a clone of the lab repository")
os.chdir(ROOT)
SUBMISSION_DIR = ROOT / "submissions" / "2A202602524_DinhBaoHung"
CODE_DIR = SUBMISSION_DIR / "code"
sys.path.insert(0, str(ROOT))
sys.path.insert(0, str(CODE_DIR))

print("python", platform.python_version(), "| torch", torch.__version__, "| timm", timm.__version__)
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "KHÔNG CÓ GPU")
if not torch.cuda.is_available():
    print("WARNING: CPU-only execution will be very slow; do not claim GPU latency results.")

### Tải dữ liệu (đã viết sẵn)
Ảnh từ Zenodo (~490 MB, kiểm tra MD5), nhãn và các file chia fold từ GitHub của tác giả.
**Sau khi giải nén, hãy `ls` để xem ảnh nằm ở thư mục nào** rồi đặt `IMAGES_DIR` cho đúng.

In [ ]:
import hashlib, urllib.request, zipfile

Path("data/labels").mkdir(parents=True, exist_ok=True)
image_files = list(Path("data").rglob("*.jpg"))
if len(image_files) != 17509:
    archive = Path("data/images.zip")
    if not archive.exists():
        urllib.request.urlretrieve("https://zenodo.org/records/7939060/files/images.zip?download=1", archive)
    checksum = hashlib.md5()
    with archive.open("rb") as source:
        for chunk in iter(lambda: source.read(1 << 20), b""):
            checksum.update(chunk)
    digest = checksum.hexdigest()
    assert digest == "b7b30f96d466fba86016aa5a26606e0f", f"MD5 sai: {digest}"
    with zipfile.ZipFile(archive) as zipped:
        zipped.extractall("data")
    image_files = list(Path("data").rglob("*.jpg"))
assert len(image_files) == 17509, f"Cần 17.509 ảnh, tìm thấy {len(image_files)}"

BASE = "https://raw.githubusercontent.com/AlexOlsen/DeepWeeds/master/labels"
for name in ["labels", "train_subset0", "val_subset0", "test_subset0"]:
    target = Path("data/labels") / f"{name}.csv"
    if not target.exists():
        urllib.request.urlretrieve(f"{BASE}/{name}.csv", target)
print("images:", len(image_files), "labels:", sorted(p.name for p in Path('data/labels').glob('*.csv')))

In [ ]:
IMAGES_DIR = "data"   # Dataset resolves filenames recursively beneath this directory.
LABELS_DIR = "data/labels"
from train import Config, run
from dataset import load_split, check_split
train_df, val_df, test_df = load_split(LABELS_DIR, fold=0)
split_summary = check_split(train_df, val_df, test_df, IMAGES_DIR)
PATH_CFG = {"images_dir": IMAGES_DIR, "labels_dir": LABELS_DIR,
            "batch_size": 8, "num_workers": 0,
            "out_dir": str(SUBMISSION_DIR / "runs"),
            "pred_dir": str(SUBMISSION_DIR / "predictions"),
            "curves_dir": str(SUBMISSION_DIR / "curves")}
for folder in (SUBMISSION_DIR, SUBMISSION_DIR / "curves", SUBMISSION_DIR / "predictions"):
    folder.mkdir(parents=True, exist_ok=True)

## Bước 0 — EDA và kiểm tra pipeline
Mục tiêu: hiểu dữ liệu, chạy các kiểm tra chia dữ liệu (README mục 2.1), kiểm tra pipeline trước khi chạy thật.

In [ ]:
import matplotlib.pyplot as plt
from PIL import Image
from dataset import CLASS_NAMES, _resolve_image

for split_name, frame in [("train", train_df), ("val", val_df), ("test", test_df)]:
    counts = frame["Label"].value_counts().reindex(range(9), fill_value=0)
    print(split_name, len(frame), dict(zip(CLASS_NAMES, counts.tolist())))
train_df["Species"].value_counts().plot(kind="bar", figsize=(11, 4), title="Fold 0 train class distribution")
plt.ylabel("Images"); plt.tight_layout(); plt.show()

fig, axes = plt.subplots(9, 3, figsize=(9, 24))
for label, class_name in enumerate(CLASS_NAMES):
    examples = train_df[train_df.Label == label].head(3)
    for column, (_, row) in enumerate(examples.iterrows()):
        with Image.open(_resolve_image(Path(IMAGES_DIR), str(row.Filename))) as image:
            axes[label, column].imshow(image.convert("RGB"))
        axes[label, column].axis("off")
        if column == 0: axes[label, column].set_ylabel(class_name)
plt.tight_layout(); plt.show()

In [ ]:
from model import build_model
from dataset import build_transforms, make_loader
from train import set_seed
import torch
set_seed(0)
probe_df = train_df.head(8).copy()
probe_loader = make_loader(probe_df, IMAGES_DIR, build_transforms(True, 224), 8, True, num_workers=0)
images, labels, names = next(iter(probe_loader))
probe_model = build_model("resnet50", pretrained=True).cuda()
probe_model.eval()
with torch.inference_mode():
    initial_loss = torch.nn.functional.cross_entropy(probe_model(images.cuda()), labels.cuda()).item()
print("initial CE (target near 2.197):", initial_loss)
probe_model.train()
optimizer = torch.optim.AdamW(probe_model.parameters(), lr=1e-4)
for _ in range(100):
    optimizer.zero_grad(set_to_none=True)
    loss = torch.nn.functional.cross_entropy(probe_model(images.cuda()), labels.cuda())
    loss.backward(); optimizer.step()
print("same-batch overfit loss:", float(loss.detach()))
assert float(loss.detach()) < 0.15, "Pipeline did not overfit the tiny batch; debug before experiments"
del probe_model, optimizer
torch.cuda.empty_cache()

## Bước 1 — So sánh backbone (≥ 5)
Cùng công thức nền `T00` (GUIDE.md mục 1.4), cùng seed, mỗi backbone một lần chạy. Mã thí nghiệm `B01`, `B02`, ...

In [ ]:
import gc, json
from benchmark import latency_report
from model import build_model
backbone_names = ["resnet50", "resnext50_32x4d", "convnext_tiny",
                  "deit_small_patch16_224", "mobilenetv3_large_100"]
backbone_results = []
for index, backbone_name in enumerate(backbone_names, start=1):
    result = run(Config(exp_id=f"B{index:02d}", backbone=backbone_name, seed=0, **PATH_CFG))
    benchmark_model = build_model(backbone_name, pretrained=True).cuda().eval()
    checkpoint = torch.load(result["checkpoint"], map_location="cuda", weights_only=False)
    benchmark_model.load_state_dict(checkpoint["model"])
    preliminary_latency = latency_report(benchmark_model, 1, 224, "fp32", "cuda", iters=50)
    result["latency_batch1_p50_ms"] = preliminary_latency["p50"]
    result["latency_batch1_p95_ms"] = preliminary_latency["p95"]
    Path(result["checkpoint"]).with_name("result.json").write_text(json.dumps(result, indent=2), encoding="utf-8")
    backbone_results.append(result)
    del result, benchmark_model; gc.collect(); torch.cuda.empty_cache()
backbone_results.sort(key=lambda row: row["macro_f1_val"], reverse=True)
print(pd.DataFrame(backbone_results)[["exp_id", "backbone", "pretrained_tag", "params_m", "gmacs", "macro_f1_val", "top1_val", "seconds_per_epoch"]])
SELECTED_BACKBONE = backbone_results[0]["backbone"]
SELECTED_BACKBONES = [row["backbone"] for row in backbone_results[:2]]
print("Selected from validation only:", SELECTED_BACKBONES)
(SUBMISSION_DIR / "backbone_results.json").write_text(json.dumps(backbone_results, indent=2), encoding="utf-8")

## Bước 2 — Công thức huấn luyện (≥ 3 trục)
Mỗi lần chạy chỉ khác `T00` **một yếu tố**. Mã `T01`, `T02`, ... Ghi rõ khác ở điểm nào.

In [ ]:
training_configs = [
    ("T00", {}),
    ("T01", {"init": "frozen"}),             # A: chỉ train head
    ("T02", {"init": "scratch"}),            # A: khởi tạo từ đầu
    ("T03", {"aug": "color"}),               # B: đổi màu
    ("T04", {"mix": "cutmix"}),              # B: CutMix
    ("T05", {"loss": "ls", "label_smoothing": 0.1}), # C: label smoothing
    ("T06", {"loss": "ce_weighted"}),         # C: CE cân bằng lớp
    ("T07", {"sampler": "balanced"}),         # D: sampler cân bằng
    ("T08", {"ema_decay": 0.999}),             # F: EMA
    ("T09", {"mix": "cutmix", "loss": "ls", "label_smoothing": 0.1, "ema_decay": 0.999}),
]
training_results = []
for exp_id, overrides in training_configs:
    result = run(Config(exp_id=exp_id, backbone=SELECTED_BACKBONE, seed=0, **overrides, **PATH_CFG))
    result["changed_from_T00"] = json.dumps(overrides, sort_keys=True) or "baseline"
    training_results.append(result)
    gc.collect(); torch.cuda.empty_cache()
baseline_f1 = next(row["macro_f1_val"] for row in training_results if row["exp_id"] == "T00")
for row in training_results:
    row["delta_vs_T00"] = row["macro_f1_val"] - baseline_f1
    row["interpretation"] = "1 seed: descriptive delta only; seed noise cannot be estimated here."
print(pd.DataFrame(training_results)[["exp_id", "changed_from_T00", "macro_f1_val", "top1_val", "delta_vs_T00"]])
(SUBMISSION_DIR / "training_results.json").write_text(json.dumps(training_results, indent=2), encoding="utf-8")

## Bước 3 — Suy luận (≥ 4 phương pháp) và độ trễ
Làm trên **val**. Không huấn luyện lại. Mã `I00` (1 view, mốc), `I01`, ...

In [ ]:
from inference import (predict_logits, view_hflip, aggregate_views, views_multicrop,
                       fit_temperature, apply_temperature)
from benchmark import latency_report, tta_latency
from dataset import build_transforms, make_loader
from eval import compute_metrics
final_recipe = max(training_results, key=lambda row: row["macro_f1_val"])
final_cfg = Config(exp_id=final_recipe["exp_id"], backbone=SELECTED_BACKBONE, seed=0,
                   **json.loads(final_recipe["changed_from_T00"]), **PATH_CFG)
final_model = build_model(final_cfg.backbone, pretrained=final_cfg.init != "scratch",
                          init=final_cfg.init).cuda()
ckpt = torch.load(SUBMISSION_DIR / "runs" / final_cfg.exp_id / "seed0" / "best_checkpoint.pt", map_location="cuda", weights_only=False)
final_model.load_state_dict(ckpt["ema"] if ckpt.get("ema") is not None else ckpt["model"])
val_loader = make_loader(val_df, IMAGES_DIR, build_transforms(False, 224), 8, False, num_workers=0)
filenames, y_val, logits, = predict_logits(final_model, val_loader, "cuda")
T = fit_temperature(logits, y_val)
print("temperature fitted on validation:", T)

inference_rows = []
base_probs = aggregate_views([logits], "prob")
flip_names, flip_y, flip_logits = predict_logits(final_model, val_loader, "cuda", view_hflip)
assert filenames == flip_names and np.array_equal(y_val, flip_y)
flip_probs = aggregate_views([logits, flip_logits], "prob")
logit_mean_probs = aggregate_views([logits, flip_logits], "logit")
amp_names, amp_y, amp_logits = predict_logits(final_model, val_loader, "cuda", amp=True)
assert filenames == amp_names and np.array_equal(y_val, amp_y)
for exp_id, method, probs, views in [("I00", "single center view", base_probs, 1),
                                     ("I02", "flip TTA: probability mean", flip_probs, 2),
                                     ("I03", "flip TTA: logit mean", logit_mean_probs, 2),
                                     ("I07", "temperature scaling", apply_temperature(logits, T), 1),
                                     ("I08", "AMP inference", aggregate_views([amp_logits], "prob"), 1)]:
    metrics = compute_metrics(y_val, probs.argmax(1), probs)
    inference_rows.append({"exp_id": exp_id, "method": method, "views": views,
                           "macro_f1_val": metrics["macro_f1"], "top1_val": metrics["top1"], "ece_val": metrics["ece"]})
latency_rows = [latency_report(final_model, 1, 224, "fp32", "cuda", iters=50),
                latency_report(final_model, 1, 224, "amp", "cuda", iters=50),
                latency_report(final_model, 8, 224, "fp32", "cuda", iters=50)]
latency_rows[0]["exp_id"] = "I00"; latency_rows[1]["exp_id"] = "I08"; latency_rows[2]["exp_id"] = "I00_batch8"
latency_rows.append({"exp_id": "I02", "method": "flip probability TTA", **tta_latency(final_model, 2, batch_size=1, img_size=224, device="cuda", iters=50)})
latency_rows.append({"exp_id": "I03", "method": "flip logit TTA", **tta_latency(final_model, 2, batch_size=1, img_size=224, device="cuda", iters=50)})
latency_by_id = {row["exp_id"]: row for row in latency_rows}
candidate_rows = []
method_map = {"I00": "single", "I02": "hflip_prob", "I03": "hflip_logit", "I07": "single", "I08": "amp"}
for row in inference_rows:
    latency = latency_by_id.get(row["exp_id"], latency_by_id["I00"])
    if latency["p95"] <= 100:
        candidate_rows.append((row, latency))
selected_inference, selected_latency = min(
    candidate_rows, key=lambda pair: (-pair[0]["macro_f1_val"], pair[1]["p95"])
)
FINAL_INFERENCE_ID = selected_inference["exp_id"]
FINAL_METHOD = method_map[FINAL_INFERENCE_ID]
print("Chosen on validation within 100 ms p95:", FINAL_INFERENCE_ID, FINAL_METHOD, selected_latency["p95"])
inference_table = pd.DataFrame(inference_rows)
latency_table = pd.DataFrame(latency_rows)
print(inference_table); print(latency_table)
fig, ax = plt.subplots(figsize=(7, 5))
for row in inference_rows:
    latency = next((item for item in latency_rows if item.get("exp_id") == row["exp_id"]), latency_rows[0])
    ax.scatter(latency["p50"], row["macro_f1_val"]); ax.annotate(row["exp_id"], (latency["p50"], row["macro_f1_val"]))
ax.set(xlabel="Latency p50, batch 1 (ms)", ylabel="Validation macro-F1", title="Accuracy-latency trade-off")
fig.tight_layout(); fig.savefig(SUBMISSION_DIR / "curves" / "inference_accuracy_latency.png", dpi=160); plt.close(fig)
(SUBMISSION_DIR / "inference_results.json").write_text(json.dumps(inference_rows, indent=2), encoding="utf-8")
(SUBMISSION_DIR / "latency_results.json").write_text(json.dumps(latency_rows, indent=2), encoding="utf-8")

## Bước 4 — Chung kết: ≥ 3 seed, test một lần mỗi seed
1. Chốt cấu hình trên **val**.
2. Chạy cấu hình cuối **và mốc** (`T00` + `I00`) với cùng số seed, bật `save_test_predictions=True`.
3. Dự đoán lưu ở `predictions/<exp_id>_seed<k>_test.csv`. Rồi chạy `eval.py` (ô dưới).

In [ ]:
from eval import save_predictions
from inference import apply_temperature
FINAL_EXP_ID = "F01uncal"
for seed in (0, 1, 2):
    final_result = run(Config(exp_id=FINAL_EXP_ID, backbone=SELECTED_BACKBONE, seed=seed,
                              **json.loads(final_recipe["changed_from_T00"]),
                              inference_method=FINAL_METHOD,
                              save_test_predictions=True, **PATH_CFG))
    baseline_result = run(Config(exp_id="T00", backbone=SELECTED_BACKBONE, seed=seed,
                                 save_test_predictions=True, **PATH_CFG))
    # Calibrate saved final logits with T fitted on validation; do not inspect test scores.
    run_folder = SUBMISSION_DIR / "runs" / FINAL_EXP_ID / f"seed{seed}"
    final_model = build_model(SELECTED_BACKBONE, pretrained=final_cfg.init != "scratch",
                              init=final_cfg.init).cuda().eval()
    final_checkpoint = torch.load(run_folder / "best_checkpoint.pt", map_location="cuda", weights_only=False)
    final_model.load_state_dict(final_checkpoint["ema"] if final_checkpoint.get("ema") is not None else final_checkpoint["model"])
    if FINAL_METHOD == "hflip_prob":
        _, _, base_seed_logits = predict_logits(final_model, val_loader, "cuda")
        _, _, flip_seed_logits = predict_logits(final_model, val_loader, "cuda", view_hflip)
        val_logits = np.log(aggregate_views([base_seed_logits, flip_seed_logits], "prob").clip(1e-12))
    elif FINAL_METHOD == "hflip_logit":
        _, _, base_seed_logits = predict_logits(final_model, val_loader, "cuda")
        _, _, flip_seed_logits = predict_logits(final_model, val_loader, "cuda", view_hflip)
        val_logits = (base_seed_logits + flip_seed_logits) / 2
    else:
        _, _, val_logits = predict_logits(final_model, val_loader, "cuda", amp=(FINAL_METHOD == "amp"))
    T_seed = fit_temperature(val_logits, y_val)
    val_probs = apply_temperature(val_logits, T_seed)
    from dataset import load_split
    _, val_df_seed, _ = load_split(LABELS_DIR, 0)
    save_predictions(SUBMISSION_DIR / "predictions" / f"F01_seed{seed}_val.csv",
                     val_df_seed.Filename, val_df_seed.Label, val_probs)
    test_logits = np.load(run_folder / "test_logits.npy")
    _, _, test_df_seed = load_split(LABELS_DIR, 0)
    test_probs = apply_temperature(test_logits, T_seed)
    save_predictions(SUBMISSION_DIR / "predictions" / f"F01_seed{seed}_test.csv",
                     test_df_seed.Filename, test_df_seed.Label, test_probs)
    print(seed, final_result["macro_f1_val"], baseline_result["macro_f1_val"], "T=", T_seed)
    del final_model; torch.cuda.empty_cache()
    gc.collect(); torch.cuda.empty_cache()

In [ ]:
# eval.py score: compute scores from final and baseline files without editing eval.py.
from eval import main as eval_main
eval_out = str(SUBMISSION_DIR / "eval_out")
assert eval_main(["score", "--pred", str(SUBMISSION_DIR / "predictions" / "F01_seed*_test.csv"),
                  "--test-csv", str(Path(LABELS_DIR) / "test_subset0.csv"),
                  "--labels", str(Path(LABELS_DIR) / "labels.csv"), "--tag", "F01", "--out", eval_out]) == 0
assert eval_main(["score", "--pred", str(SUBMISSION_DIR / "predictions" / "T00_seed*_test.csv"),
                  "--test-csv", str(Path(LABELS_DIR) / "test_subset0.csv"),
                  "--labels", str(Path(LABELS_DIR) / "labels.csv"), "--tag", "T00", "--out", eval_out]) == 0

In [ ]:
# eval.py grade: final calibrated predictions vs. uncalibrated and T00 baseline.
best_p95 = selected_latency["p95"]
assert eval_main(["grade", "--final", str(SUBMISSION_DIR / "predictions" / "F01_seed*_test.csv"),
                  "--baseline", str(SUBMISSION_DIR / "predictions" / "T00_seed*_test.csv"),
                  "--uncal", str(SUBMISSION_DIR / "predictions" / "F01uncal_seed*_test.csv"),
                  "--final-val", str(SUBMISSION_DIR / "predictions" / "F01_seed*_val.csv"),
                  "--latency-p95-ms", str(best_p95), "--latency-method", "proper",
                  "--test-csv", str(Path(LABELS_DIR) / "test_subset0.csv"),
                  "--val-csv", str(Path(LABELS_DIR) / "val_subset0.csv"),
                  "--labels", str(Path(LABELS_DIR) / "labels.csv"), "--out", eval_out]) == 0

## Bước 5 — Sản phẩm
`results.xlsx` (các sheet ở GUIDE.md mục 6.1), `curves/<exp_id>_*.png`, `report.md`, `predictions/`, `code/`.

In [ ]:
from submission_tools import build_results_workbook, write_submission_docs, validate_submission
build_results_workbook(SUBMISSION_DIR)
write_submission_docs(SUBMISSION_DIR)
validation = validate_submission(SUBMISSION_DIR, require_predictions=True)
print(validation)